# Inspecting the transformed `stops` table

Built by `priorbahn/data/prep.py` from the August and September raw files (rules in `docs/data-prep-plan.md`). Run the script first: `uv run --no-sync python -m priorbahn.data.prep`.

The table has about 28M rows and takes a few GB of memory in pandas. Some cells (sections 4 and 7) need a minute.

Sections:
1. Columns and first look
2. What was dropped
3. Splits
4. Run integrity (missing stops in the source)
5. Delay targets over time
6. Which schedule columns carry signal
7. Is there history for the validation week?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 40)

stops = pd.read_parquet("../data/processed/stops.parquet")

## 1. Columns and first look

One row is one stop event of one run (one trip of one train on one day).

| Column | Meaning |
|---|---|
| `run_id`, `run_day` | The run, and the day it started (the unit used for splitting) |
| `split` | `context` (Aug 1 to Sep 16), `validation` (Sep 17 to 23), `test` (Sep 24 to 30) |
| `train_type`, `train_key`, `line_number`, `final_destination` | Train type, `train_type + train_number`, line label, last station (the source leaves it empty at the last stop, where we fill in `station`) |
| `station`, `prev_station` | Station name (EVA code if it has no name), and the previous stop of the run |
| `stop_num`, `n_stops`, `stop_frac` | Stop number, highest stop number seen in the run, and their ratio |
| `planned_arr`, `planned_dep` | Scheduled times (missing at the first and last stop) |
| `arr_canceled`, `dep_canceled` | Cancel flags |
| `arr_delay`, `dep_delay` | Targets in minutes, recomputed from the time columns, missing when canceled |
| `dwell_planned_min`, `run_planned_min` | Planned time standing at the stop, and planned time since the previous stop |
| `arr_hour`, `arr_minute`, `dep_hour`, `dep_minute`, `weekday` | Hour and minute of the planned arrival and departure (missing at the first and last stop), and the weekday of the run start (1 is Monday) |

No column holds an actual time, so no feature leaks the target.

In [ ]:
print(f"{len(stops):,} rows")
stops.head(8)

In [ ]:
# Share of missing values per column. Missing values are expected for the first/last stop
# and for canceled events.
stops.isna().mean().rename("missing_share").to_frame()

## 2. What was dropped

The raw files are filtered step by step, in the same order as in `priorbahn/data/prep.py`. The last row must match the row count above.

In [ ]:
columns = [
    "id",
    "train_type",
    "is_replacement_train",
    "is_additional_stop",
    "arrival_planned_time",
    "departure_planned_time",
]
raw = pd.concat(
    [
        pd.read_parquet(
            f"../data/monthly_processed_data/data-2026-{m}.parquet", columns=columns
        )
        for m in ("08", "09")
    ],
    ignore_index=True,
)
run_day = pd.to_datetime(
    raw["id"].str.extract(r"-(\d{6})\d{4}-\d+$")[0], format="%y%m%d"
)
is_bus = raw["train_type"].str.contains("^(?:bus|sev|bsv)$", case=False)

# each step is a True/False column: True means the row is kept by that step
steps = [
    ("raw rows", pd.Series(True, index=raw.index)),
    (
        "has a train type, and is not a bus or rail replacement bus",
        raw["train_type"].notna() & ~is_bus,
    ),
    ("not a replacement train", ~raw["is_replacement_train"]),
    ("not an additional stop", ~raw["is_additional_stop"]),
    (
        "has a planned time",
        raw["arrival_planned_time"].notna() | raw["departure_planned_time"].notna(),
    ),
    ("run started Aug 1 to Sep 30", run_day.between("2026-08-01", "2026-09-30")),
]

kept = pd.Series(True, index=raw.index)
rows = []
for label, keeps_row in steps:
    kept = kept & keeps_row  # a row survives only if every step so far kept it
    rows.append((label, kept.sum()))
drops = pd.DataFrame(rows, columns=["after step", "rows"])
drops["dropped"] = drops["rows"].diff()
drops

## 3. Splits

Split by the day a run starts. A run belongs to exactly one split.

In [ ]:
stops.groupby("split").agg(
    stop_events=("run_id", "size"),
    runs=("run_id", "nunique"),
    first_day=("run_day", "min"),
    last_day=("run_day", "max"),
).sort_values("first_day")

## 4. Run integrity

The source has stops missing from some runs, and that is already the case in the raw files. Two kinds:
- **Leading missing**: the first stop we see is not stop 1.
- **Internal missing**: stop numbers are skipped inside the run. Then the previous row is not the previous stop, so `prev_station` and `run_planned_min` are left empty for that row.

`n_stops` is only the highest stop number we saw, so for runs that lose their last stops it is too small.

In [ ]:
# One row per run.
runs = stops.groupby("run_id").agg(
    n=("stop_num", "size"),
    first_num=("stop_num", "min"),
    last_num=("stop_num", "max"),
    train_type=("train_type", "first"),
)
runs["has_leading_missing"] = runs["first_num"] > 1
# stops we should see between the first and last stop number, minus the ones we do see
runs["has_internal_gap"] = (runs["last_num"] - runs["first_num"] + 1 - runs["n"]) > 0

print(runs[["has_leading_missing", "has_internal_gap"]].mean())

by_type = runs.groupby("train_type").agg(
    runs=("n", "size"),
    leading_missing=("has_leading_missing", "mean"),
    internal_gap=("has_internal_gap", "mean"),
)
by_type.sort_values("runs", ascending=False).head(12)

In [ ]:
# Stops per run, and planned times in minutes.
print(runs["n"].describe())
stops[["run_planned_min", "dwell_planned_min"]].describe(percentiles=[0.5, 0.99])

## 5. Delay targets over time

Canceled events have missing delays and are left out of these averages. The cancel rate is per event.

In [ ]:
def p90(values: pd.Series) -> float:
    return values.quantile(0.9)


stops.groupby("split").agg(
    stop_events=("run_id", "size"),
    dep_mean=("dep_delay", "mean"),
    dep_median=("dep_delay", "median"),
    dep_p90=("dep_delay", p90),
    arr_mean=("arr_delay", "mean"),
    arr_p90=("arr_delay", p90),
    dep_cancel_rate=("dep_canceled", "mean"),
).sort_index()

In [ ]:
daily = stops.groupby("run_day").agg(
    dep_mean=("dep_delay", "mean"),
    arr_mean=("arr_delay", "mean"),
    cancel_rate=("dep_canceled", "mean"),
)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.plot(daily.index, daily["dep_mean"], label="departure delay")
ax1.plot(daily.index, daily["arr_mean"], label="arrival delay")
ax1.set_ylabel("mean delay (min)")
ax1.legend()
ax2.plot(daily.index, daily["cancel_rate"], color="tab:red")
ax2.set_ylabel("cancel rate")
for ax in (ax1, ax2):
    for edge in ("2026-09-17", "2026-09-24"):  # validation and test start
        ax.axvline(pd.Timestamp(edge), color="gray", linestyle=":")
plt.tight_layout()
plt.show()

## 6. Which schedule columns carry signal

Mean departure delay by single columns, over the whole table. A flat bar chart means that column alone says little. This is a first look, not a model.

In [ ]:
def mean_by(key: pd.Series) -> pd.DataFrame:
    """Mean departure delay and row count for each value of `key`."""
    return stops.groupby(key)["dep_delay"].agg(["mean", "size"])


# stop_frac is 0 to 1, so this puts each stop into one of 10 equal slices of its run
stop_frac_decile = (
    np.ceil(stops["stop_frac"] * 10).clip(1, 10).rename("stop_frac_decile")
)

panels = {
    "dep_hour": mean_by(stops["dep_hour"]),
    "weekday": mean_by(stops["weekday"]),
    "stop_frac_decile": mean_by(stop_frac_decile),
    "train_type": mean_by(stops["train_type"])
    .sort_values("size", ascending=False)
    .head(12),
}

fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, (name, table) in zip(axes.flat, panels.items()):
    ax.bar(table.index.astype(str), table["mean"])
    ax.set_title(f"mean departure delay by {name}")
    ax.tick_params(axis="x", labelrotation=60)
plt.tight_layout()
plt.show()

## 7. Is there history for the validation week?

Whether lagged history features or retrieved context rows can work depends on how often a validation row's train and station was seen in the context pool. For each validation stop, count the distinct context days with the same `train_key` and `station`.

In [ ]:
context = stops[stops["split"] == "context"]

# distinct days on which each (train, station) pair was seen: drop repeated
# (train, station, day) rows, then count what is left per pair
seen = context[["train_key", "station", "run_day"]].drop_duplicates()
context_days = (
    seen.groupby(["train_key", "station"]).size().rename("ctx_days").reset_index()
)

# attach that count to every validation row, 0 if the pair never appeared in the context
validation = stops[stops["split"] == "validation"].merge(
    context_days, on=["train_key", "station"], how="left"
)
validation["ctx_days"] = validation["ctx_days"].fillna(0)

pd.Series(
    {f"seen_on_{k}+_days": (validation["ctx_days"] >= k).mean() for k in (1, 5, 20, 40)}
).rename(f"share of {len(validation):,} validation rows").to_frame()

In [ ]:
# The same by train type: which types have little history?
validation["seen_1plus"] = validation["ctx_days"] >= 1
validation["seen_20plus"] = validation["ctx_days"] >= 20
validation.groupby("train_type").agg(
    rows=("ctx_days", "size"),
    seen_1plus=("seen_1plus", "mean"),
    seen_20plus=("seen_20plus", "mean"),
    median_days=("ctx_days", "median"),
).sort_values("rows", ascending=False).head(12)